Ячейка 1 (Импорты и генерация данных):

In [1]:
import pandas as pd
import duckdb
import numpy as np
import time
import tracemalloc
import os
from datetime import datetime, timedelta

# Генерируем синтетический лог на 100 000 строк (для честных замеров)
np.random.seed(42)
n_rows = 100_000
base_time = datetime(2025, 1, 1)
print(f"Генерация {n_rows} строк...")

data = {
    "timestamp": [base_time + timedelta(seconds=i) for i in range(n_rows)],
    "event_type": np.random.choice(["login_success", "login_failed", "logout", "sudo"], size=n_rows),
    "user": np.random.choice(["alice", "bob", "charlie", "david"], size=n_rows),
    "severity": np.random.choice(["low", "medium", "high"], size=n_rows)
}
df = pd.DataFrame(data)
df.to_csv("individual_log.csv", index=False)

file_size_mb = os.path.getsize("individual_log.csv") / (1024 * 1024)
print(f"Файл создан: {n_rows} строк, {file_size_mb:.2f} МБ")
df.head()

Генерация 100000 строк...
Файл создан: 100000 строк, 4.01 МБ


,timestamp,event_type,user,severity
0,2025-01-01 00:00:00,logout,alice,medium
1,2025-01-01 00:00:01,sudo,david,medium
2,2025-01-01 00:00:02,login_success,bob,high
3,2025-01-01 00:00:03,logout,david,low
4,2025-01-01 00:00:04,logout,bob,low


Ячейка 2 (Задача на Pandas):

In [2]:
def agg_pandas():
    # Копируем, чтобы не портить исходный df
    d = df.copy()
    d["hour"] = d["timestamp"].dt.floor("h")
    return d.groupby(["hour", "event_type"]).size().reset_index(name="count")

Ячейка 3 (Задача на DuckDB):

In [3]:
def agg_duckdb():
    q = """
    SELECT date_trunc('hour', timestamp) AS hour,
           event_type,
           COUNT(*) AS count
    FROM read_csv_auto('individual_log.csv')
    GROUP BY hour, event_type
    ORDER BY hour, event_type
    """
    return duckdb.sql(q).df()

Ячейка 4 (Замеры и таблица):

In [4]:
def measure(fn, label, runs=3):
    times = []
    peak_mem = 0
    
    for _ in range(runs):
        tracemalloc.start()
        t0 = time.perf_counter()
        fn()
        t1 = time.perf_counter()
        _, peak = tracemalloc.get_traced_memory()
        tracemalloc.stop()
        
        times.append(t1 - t0)
        peak_mem = max(peak_mem, peak) # берем максимальную память за прогоны
        
    return {
        "Инструмент": label,
        "Время (медиана), с": round(np.median(times), 4),
        "Пиковая память (tracemalloc), МБ": round(peak_mem / 1024**2, 2)
    }

print("Замер Pandas...")
res_pd = measure(agg_pandas, "Pandas")
print("Замер DuckDB...")
res_db = measure(agg_duckdb, "DuckDB")

results_df = pd.DataFrame([res_pd, res_db])
display(results_df)

# Проверка, что результаты агрегаций совпадают
res_pd_df = agg_pandas()
res_db_df = agg_duckdb()
assert res_pd_df.shape == res_db_df.shape, "Размеры результатов не совпадают!"
print("\nРезультаты агрегаций совпадают ✓")

Замер Pandas...
Замер DuckDB...


,Инструмент,"Время (медиана), с","Пиковая память (tracemalloc), МБ"
0,Pandas,0.0424,9.76
1,DuckDB,0.4787,0.02



Результаты агрегаций совпадают ✓
